In [1]:
# %pip install opencv-python scikit-image

In [2]:
import numpy as np
import os
import cv2
from skimage.feature import hog

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [3]:
DATASET_PATH = "../data/hiragana"
IMG_SIZE = (84, 83)

In [4]:
def extract_hog(img):
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )
    return features

In [5]:
data = []
labels = []
label_names = []

for i, label in enumerate(sorted(os.listdir(DATASET_PATH))):
    folder_path = os.path.join(DATASET_PATH, label)
    
    if not os.path.isdir(folder_path):
        continue
    
    label_names.append(label)
    
    for file in os.listdir(folder_path):
        if not file.endswith(".jpg"):
            continue
        
        img_path = os.path.join(folder_path, file)
        
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, IMG_SIZE)
        
        features = extract_hog(img)
        
        data.append(features)
        labels.append(i)

X = np.array(data)
y = np.array(labels)

print("Dataset shape:", X.shape)
print("Number of classes:", len(label_names))

Dataset shape: (4600, 2916)
Number of classes: 46


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

In [8]:
param_grid = {
    "svm__C": [1, 10, 50],
    "svm__gamma": ["scale", 0.01, 0.001],
    "svm__kernel": ["rbf"]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


KeyboardInterrupt: 

In [ ]:
y_pred = grid.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_names))

Accuracy: 0.9641304347826087
              precision    recall  f1-score   support

          aa       0.95      0.90      0.92        20
         chi       1.00      1.00      1.00        20
          ee       0.39      1.00      0.56        20
          fu       1.00      0.85      0.92        20
          ha       1.00      0.95      0.97        20
          he       1.00      1.00      1.00        20
          hi       1.00      1.00      1.00        20
          ho       0.95      1.00      0.98        20
          ii       1.00      1.00      1.00        20
          ka       1.00      0.95      0.97        20
          ke       1.00      0.85      0.92        20
          ki       1.00      0.95      0.97        20
          ko       1.00      1.00      1.00        20
          ku       1.00      0.95      0.97        20
          ma       1.00      0.95      0.97        20
          me       1.00      1.00      1.00        20
          mi       1.00      0.90      0.95        2

In [ ]:
def predict_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, IMG_SIZE)
    
    features = extract_hog(img).reshape(1, -1)
    
    pred = grid.predict(features)[0]
    
    print("Prediction:", label_names[pred])

# Example
predict_image("test.jpg")

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'
